<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보충 코드, 저자: <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 임베딩 레이어(Embedding Layers)와 선형 레이어(Linear Layers)의 차이점 이해하기

- PyTorch의 임베딩 레이어는 행렬 곱셈을 수행하는 선형 레이어와 동일한 결과를 달성합니다. 임베딩 레이어를 사용하는 이유는 계산 효율성 때문입니다
- PyTorch의 코드 예제를 사용하여 이러한 관계를 단계별로 살펴보겠습니다

In [1]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.3.1


<br>
&nbsp;

## nn.Embedding 사용하기

In [2]:
# LLM 컨텍스트에서 토큰 ID를 나타낼 수 있는
# 다음 3개의 훈련 예제가 있다고 가정합니다
idx = torch.tensor([2, 3, 1])

# 임베딩 행렬의 행 수는
# 가장 큰 토큰 ID + 1로 결정할 수 있습니다.
# 가장 높은 토큰 ID가 3이라면, 가능한 토큰 ID 0, 1, 2, 3에 대해
# 4개의 행이 필요합니다
num_idx = max(idx)+1

# 원하는 임베딩 차원은 하이퍼파라미터입니다
out_dim = 5

- 간단한 임베딩 레이어를 구현해봅시다:

In [3]:
# 임베딩 레이어의 가중치는
# 작은 랜덤값으로 초기화되기 때문에
# 재현 가능성을 위해 랜덤 시드를 사용합니다
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

선택적으로 임베딩 가중치를 살펴볼 수 있습니다:

In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

- 그런 다음 임베딩 레이어를 사용하여 ID가 1인 훈련 예제의 벡터 표현을 얻을 수 있습니다:

In [5]:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- 아래는 내부적으로 일어나는 일의 시각화입니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/1.png" width="400px">

- 마찬가지로, 임베딩 레이어를 사용하여 ID가 2인 훈련 예제의 벡터 표현을 얻을 수 있습니다:

In [6]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/2.png" width="400px">

- 이제 이전에 정의한 모든 훈련 예제를 변환해보겠습니다:

In [7]:
idx = torch.tensor([2, 3, 1])
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- 내부적으로는 여전히 동일한 룩업(look-up) 개념입니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/3.png" width="450px">

<br>
&nbsp;

## nn.Linear 사용하기

- 이제 위의 임베딩 레이어가 PyTorch에서 원-핫 인코딩 표현에 대한 `nn.Linear` 레이어와 정확히 동일한 작업을 수행한다는 것을 보여드리겠습니다
- 먼저 토큰 ID를 원-핫 표현으로 변환해봅시다:

In [8]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

- 다음으로, 행렬 곱셈 $X W^\top$을 수행하는 `Linear` 레이어를 초기화합니다:

In [9]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

- PyTorch의 선형 레이어도 작은 랜덤 가중치로 초기화됩니다. 위의 `Embedding` 레이어와 직접 비교하려면 동일한 작은 랜덤 가중치를 사용해야 하므로 여기서 재할당합니다:

In [10]:
linear.weight = torch.nn.Parameter(embedding.weight.T)

- 이제 입력의 원-핫 인코딩 표현에 대해 선형 레이어를 사용할 수 있습니다:

In [11]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

보시다시피, 이는 임베딩 레이어를 사용했을 때 얻은 것과 정확히 동일합니다:

In [12]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

- 첫 번째 훈련 예제의 토큰 ID에 대해 내부적으로 일어나는 계산은 다음과 같습니다:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/4.png" width="450px">

- 그리고 두 번째 훈련 예제의 토큰 ID에 대해서는:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/embeddings-and-linear-layers/5.png" width="450px">

- 각 원-핫 인코딩된 행에서 하나의 인덱스를 제외한 모든 인덱스가 0이므로(설계상), 이 행렬 곱셈은 본질적으로 원-핫 요소의 룩업과 동일합니다
- 원-핫 인코딩에 대한 이러한 행렬 곱셈 사용법은 임베딩 레이어 룩업과 동일하지만, 대형 임베딩 행렬로 작업할 때는 0과의 낭비적인 곱셈이 많기 때문에 비효율적일 수 있습니다